In [1]:
import sys
import os
project_root = '/work'

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pandas as pd
import re
from scripts.utils.utils import normalize_text

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [4]:
dataset=pd.read_csv("/work/data/processed/extracted_features.csv")
df=dataset.copy()
df.dropna(subset=["cpu"], inplace=True)

In [5]:
def save_amd(row):
    row=row.apply(lambda x: normalize_text(x) if isinstance(x, str) else x)
    cpu = str(row['cpu'])
    name = str(row['name'])

    if len(cpu.split()) > 3: 
        return cpu
        
    match = re.search(r'ryzen\s*(?:al|ai|al\s*|\s*)([579])\s*[-]*\s*(\d{3,4}|hx\s*\d{3})', name, re.IGNORECASE)
    if match:
        seri = match.group(1)
        model = match.group(2).replace(" ", "").lower() 
        
        if len(model) == 3 or 'hx' in model:
            return f"amd ryzen ai {seri} {model}"
        else:
            return f"amd ryzen {seri} {model}"
            
    return cpu

df['cpu'] = df.apply(save_amd, axis=1)



In [6]:
df["cpu"] = df["cpu"].str.replace(r'^celeron', 'intel celeron', regex=True)
df['cpu'] = df['cpu'].str.replace(r'1334$', '1334u', regex=True)
df['cpu'] = df['cpu'].str.replace(r'1355$', '1355u', regex=True)
df['cpu'] = df['cpu'].str.replace(r'12450$', '12450h', regex=True)
df['cpu'] = df['cpu'].str.replace(r'12650$', '12650h', regex=True)
df["cpu"]= df["cpu"].str.replace('™', '').str.replace('®', '')
df["cpu"]=df["cpu"].str.replace(r'([a-zA-Z])(\d)', r'\1 \2', regex=True)
df["cpu"]=df["cpu"].str.replace(r'(\br\b)', r'ryzen', regex=True)
df["cpu"]=df["cpu"].str.replace(r"^m\s(\d)", r"apple m\1", regex=True).replace(r"apple m\s(\d)", r"apple m\1", regex=True)

df["cpu"]=df["cpu"].str.replace(r"^ryzen\b",r"amd ryzen",regex=True)
df["cpu"]=df["cpu"].str.replace(r'(\bi\b)\s(\d)', r'i\2', regex=True)
df["cpu"]=df["cpu"].str.replace(r'(\bu\b)\s(\d)', r'ultra \2', regex=True)
df["cpu"]=df["cpu"].str.replace("ryzenai\b", "ryzen ai", regex=True)

In [7]:
intel_pattern = re.compile(
    r"""
    \b(intel\s+|core\s+|ultra\s+|i\s*)  # GRUP 1: KESİNLİKLE bunlardan biriyle başlamak ZORUNDA!
    (?:intel\s+|core\s+|ultra\s+|i\s*)* # GRUP YOK: Satıcı 2-3 kere core, intel yazdıysa onları yut
    ([3579])                            # GRUP 2: İşlemci Ailesi (3, 5, 7, 9)
    \s*
    (\d{3,5})                           # GRUP 3: Model Numarası (3, 4, 5 hane)
    \s*
    ([a-z]{1,2})?\b                     # GRUP 4: Suffix (u, h, hx vb.)
    """,
    re.IGNORECASE | re.VERBOSE
)

def standardize_intel(match):
    # Tüm eşleşen metni küçük harfe çevirip alalım ki içinde 'ultra' var mı bakalım
    tam_eslesme = match.group(0).lower()
    is_ultra = "ultra" in tam_eslesme
    
    tier = match.group(2)
    model = match.group(3)
    suffix = match.group(4) or ""
    
    # ALTIN KURAL 1: Model 3 haneli ise YENİ NESİLDİR (i takısı almaz)
    if len(model) == 3:
        if is_ultra:
            return f"intel core ultra {tier} {model}{suffix}"
        else:
            return f"intel core {tier} {model}{suffix}"
            
    # ALTIN KURAL 2: Model 4 veya 5 haneli ise ESKİ NESİLDİR (i takısı alır)
    else:
        return f"intel core i{tier} {model}{suffix}"

In [8]:
df["cpu"] = df["cpu"].apply(lambda x: intel_pattern.sub(standardize_intel, x))

In [9]:
df["cpu"]=df["cpu"].str.replace(r"^i(\d)", r"intel i\1",regex=True)
df["cpu"]=df["cpu"].str.replace(r"^u(\d)", r"ultra \1",regex=True)
df["cpu"]=df["cpu"].str.replace(r"^core\b", r"intel core", regex=True)
df["cpu"]=df["cpu"].str.replace(r"\bultra\s(\d)\b", r"intel ultra \1", regex=True)
df["cpu"]=df["cpu"].str.replace(r"intel\s(i\d)",r"intel core \1",regex=True)
df["cpu"]=df["cpu"].str.replace("snapdragonx", "snapdragon x", regex=True)

df["cpu"] = df["cpu"].str.replace(r'intelcore\s*intel core', 'intel core', regex=True)
df["cpu"] = df["cpu"].str.replace('intelcore', 'intel core')
df["cpu"] = df["cpu"].str.replace('ryzenai', 'amd ryzen ai')
df["cpu"] = df["cpu"].str.replace(r'g\s+(\d)', r'g\1', regex=True)     
df["cpu"] = df["cpu"].str.replace(r'hx\s+(\d)', r'hx\1', regex=True)   
df["cpu"] = df["cpu"].str.replace(r'z\s+(\d)', r'z\1', regex=True)
df["cpu"] = df["cpu"].str.replace(r'n\s+(\d)', r'n\1', regex=True)     
df["cpu"] = df["cpu"].str.replace(r'snapdragonx?\s*x\s*1', 'snapdragon x1', regex=True)
df["cpu"] = df["cpu"].str.replace(r'^ultra', 'intel ultra', regex=True)
df["cpu"] = df["cpu"].str.strip()

In [10]:
df["extract_cpu_brand"]=df["cpu"].str.extract(r'^(intel core|intel ultra|amd ryzen|apple|snapdragon)', expand=False)
df["cpu_ryzen_series"]=df["cpu"].str.extract(r'ryzen\s*(\d+)', expand=False)
df["intel_core_series"]=df["cpu"].str.extract("i(\d)", expand=False)
df["cpu_gen"] = df["cpu"].str.extract(
    r'(\d{3,5}(?:hx|h|u|v|k|f)?|[NJ]\d{4})',
    expand=False
)

In [11]:
df[df["cpu_gen"].isnull()]

,brand,name,price,cpu,gpu,storage,ram,os,extract_cpu_brand,cpu_ryzen_series,intel_core_series,cpu_gen
21,ASUS,"Vivobook 16 X1607QA-MB085W/ Snapdragon X1/ 16 GB Ram/ 512 GB SSD/ 16""/ W11 Laptop Soğuk Gümüş",33500.00,snapdragon x1,NaN,512 GB,16 GB,Windows 11,snapdragon,NaN,NaN,NaN
45,ASUS,"Zenbook 14 UX3407QA-QD381W Snapdragon 16GB LPDDR5X 512GB SSD 14"" 60hz OLED W11 Home Laptop",44499.00,snapdragon16gb,NaN,512 GB,16 GB,Windows 11 Home,snapdragon,NaN,NaN,NaN
72,ASUS,"VivoBook 16 X1607QA-MB052W SnapdragonX X1-26-100 16GB 512SSD 16"" WUXGA W11H Dizüstü Bilgisayar",30999.00,snapdragon x x 1,NaN,512 GB,16 GB,Windows 11 Home,snapdragon,NaN,NaN,NaN
88,ASUS,"ROG Xbox Ally RC73YA 7 inç FHD (1920 x 1080) 16:9 120Hz, Ryzen™ AI Z2 A, 16GB , 512GB PCIE M.2.",32999.00,amd ryzen ai z2,NaN,512 GB,16 GB,NaN,amd ryzen,NaN,NaN,NaN
123,ASUS,ROG Xbox Ally X RC73XA-ROG Ally AMD Ryzen AI Z2 24GB RAM 1TB SSD 7'' FHD Windows 11 Home,61999.00,amd ryzen ai z2,NaN,1 TB,24 GB,Windows 11 Home,amd ryzen,NaN,NaN,NaN
141,ASUS,"X540ba-dm317 A6-9225 4gb 256gb Ssd 15.6"" Dos",22999.00,apple m3,NaN,256 GB,4 GB,FreeDOS,apple,NaN,NaN,NaN
522,ASUS,"VivoBook 16 X1607QA-MB089W001 Snapdragon X X1-26-100 16GB 2TBSSD 16"" WUXGA W11H Dizüstü Bilgisayar",42775.00,snapdragon x x 1,NaN,2 TB,16 GB,Windows 11 Home,snapdragon,NaN,NaN,NaN
1320,ASUS,"VivoBook 16 X1607QA-MB089W002 Snapdragon X X1-26-100 16GB 1TBSSD 16"" WUXGA W11H Dizüstü Bilgisa",43154.00,snapdragon x x 1,NaN,1 TB,16 GB,Windows 11 Home,snapdragon,NaN,NaN,NaN
2503,ASUS,"PM3406CKA AI R5-330 40GB 512 GB SSD 14"" WUXGA FingerPrint, Aluminum Case Freedos R516512G0D-47",58780.00,amd ryzen5,NaN,512 GB,40 GB,FreeDOS,amd ryzen,5,NaN,NaN
2524,ASUS,"PM3406CKA AI R5-330 40GB 512 GB SSD 14"" WUXGA FingerPrint, Aluminum Case Freedos R516512G0D-47",58780.00,amd ryzen5,NaN,512 GB,40 GB,FreeDOS,amd ryzen,5,NaN,NaN


In [12]:
print(df["cpu"].unique())


['amd ryzen7 7445hs' 'intel core 5 120u' 'intel celeron n4500'
 'intel core i5 13450hx' 'amd ryzen5 7520u' 'intel celeron n4020'
 'amd ryzen7 7445' 'amd ryzen5 7520' 'intel core 5 210h'
 'intel core i7 14650hx' 'amd ryzen9 9955' 'amd ryzen5 7535hs'
 'snapdragon x1' 'intel core i5 1135g7' 'intel core i5 1334u'
 'amd ryzen9 8940hx' 'intel core i5 13420h' 'amd ryzen7 5825u'
 'intel core i7 13620h' 'snapdragon16gb' 'intel core 5 120'
 'amd ryzen7 5825' 'amd ryzen ai 9 270' 'intel core intel ultra 9 275hx'
 'intel core intel ultra 7 255h' 'intel core intel ultra 7 255hx'
 'snapdragon x x 1' 'intel core i3 1315u' 'amd ryzen5 7535'
 'intel core intel ultra 9 285hx' 'intel core intel ultra 9 285h'
 'amd ryzen ai z2' 'amd ryzen ai 9 hx370' 'amd ryzen9 8940'
 'intel core i5 1335u' 'amd ryzen ai 7 260' 'intel core i9 14900hx'
 'intel core 7 150u' 'intel core intel ultra 7 258v'
 'intel core i5 12500h' 'amd ryzen ai 7 350' 'amd ryzen7 7735hs'
 'apple m3' 'intel core intel ultra 5 120u' 'amd ryzen7

In [13]:
cpu_mapping = {
    # ── Tier 1: Giriş seviyesi / Eski ──
    'intel celeron n4020': 1, 'intel celeron n4120': 1, 'intel celeron n4500': 1,
    'intel core i3 1005g1': 1, 'intel core i3 1115g4': 1,
    'intel core i5 1035g1': 1, 'intel core i5 8250u': 1,
    'intel core 3 100u': 1,
    'amd ryzen 3 7320u': 1, 'amd ryzen 5 3500u': 1,

    # ── Tier 2: Orta-alt ──
    'intel core i3 1215u': 2, 'intel core i3 1305u': 2, 'intel core i3 1315u': 2,
    'intel core i5 1135g7': 2, 'intel core i5 1145g7': 2,
    'intel core i5 1235u': 2, 'intel core i5 1240p': 2,
    'intel core i5 1334u': 2, 'intel core i5 1335u': 2, 'intel core i5 1335': 2,
    'intel core i5 10310u': 2,
    'intel core i7 1165g7': 2, 'intel core i7 1255u': 2, 'intel core i7 1260p': 2,
    'intel core i7 1355u': 2, 'intel core i7 1335u': 2,
    'intel core 5 120u': 2, 'intel core 5 120': 2,
    'amd ryzen 5 5500u': 2, 'amd ryzen 5 5500': 2,
    'amd ryzen 5 5625u': 2, 'amd ryzen 5 5625': 2,
    'amd ryzen 5 6600': 2,
    'amd ryzen 7 5700u': 2, 'amd ryzen 7 5700': 2,
    'amd ryzen 7 5825u': 2, 'amd ryzen 7 5825': 2,

    # ── Tier 3: Orta ──
    'intel core i5 12450h': 3, 'intel core i5 12450hx': 3,
    'intel core i5 12500h': 3, 'intel core i5 12600hx': 3,
    'intel core i7 12650h': 3, 'intel core i7 12650hx': 3,
    'intel core i7 12700h': 3,
    'intel core i9 1490hx': 3,
    'intel core 5 210h': 3, 'intel core 7 150u': 3,
    'intel core i5 13420h': 3, 'intel core i5 13450hx': 3,
    'intel core i5 13500h': 3,
    'amd ryzen 5 7235': 3, 'amd ryzen 5 7430': 3, 'amd ryzen 5 7430u': 3,
    'amd ryzen 5 7520u': 3, 'amd ryzen 5 7520': 3,
    'amd ryzen 5 7530u': 3, 'amd ryzen 5 7530': 3,
    'amd ryzen 7 7435': 3, 'amd ryzen 7 7435hs': 3,
    'amd ryzen 7 7445': 3, 'amd ryzen 7 7445h': 3, 'amd ryzen 7 7445hs': 3,
    'amd ryzen 7 7730u': 3, 'amd ryzen 7 7730': 3,
    'apple m1': 3,

    # ── Tier 4: Orta-üst ──
    'intel core i5 14450hx': 4, 'intel core i5 14500hx': 4,
    'intel core i7 13620h': 4, 'intel core i7 13650hx': 4,
    'intel core i7 13700h': 4, 'intel core i7 13700hx': 4,
    'intel core i7 13850hx': 4,
    'intel core i9 13900h': 4, 'intel core i9 13900hx': 4,
    'intel core i9 13980hx': 4,
    'intel core 5 235u': 4, 'intel core 7 240h': 4, 'intel core 7 250h': 4,
    'intel core 7 265u': 4, 'intel core 9 270h': 4, 'intel core 9 275hx': 4,
    'intel core intel ultra 5 112u': 4, 'intel core intel ultra 5 120u': 4,
    'intel core intel ultra 5 125h': 4, 'intel core intel ultra 5 125u': 4,
    'intel core intel ultra 7 150u': 4, 'intel core intel ultra 7 155h': 4,
    'intel core intel ultra 7 155u': 4,
    'amd ryzen 5 7535hs': 4, 'amd ryzen 5 7535': 4,
    'amd ryzen 5 7640hs': 4, 'amd ryzen 5 7640': 4,
    'amd ryzen 7 7735': 4, 'amd ryzen 7 7735hs': 4,
    'amd ryzen 7 7840': 4, 'amd ryzen 7 7840hs': 4,
    'amd ryzen 9 9955': 4,
    'apple m2': 4,'snapdragon x1': 4,

    # ── Tier 5: Üst ──
    'intel core i7 14650hx': 5, 'intel core i7 14700hx': 5,
    'intel core i7 11800h': 5,
    'intel core i9 14900hx': 5,
    'intel core intel ultra 5 210h': 5, 'intel core intel ultra 5 225h': 5,
    'intel core intel ultra 5 225u': 5, 'intel core intel ultra 5 226v': 5,
    'intel core intel ultra 5 228v': 5, 'intel core intel ultra 5 235u': 5,
    'intel core intel ultra 5 238v': 5, 'intel core intel ultra 5 255h': 5,
    'intel core intel ultra 5 255u': 5,
    'intel core intel ultra 7 240h': 5, 'intel core intel ultra 7 255h': 5,
    'intel core intel ultra 7 255hx': 5, 'intel core intel ultra 7 255u': 5,
    'intel core intel ultra 7 256v': 5, 'intel core intel ultra 7 258v': 5,
    'intel core intel ultra 7 265h': 5, 'intel core intel ultra 7 265u': 5,
    'intel core intel ultra 7 268v': 5,
    'intel core intel ultra 9 270h': 5,
    'amd ryzen 5 8645hs': 5, 'amd ryzen 5 8645': 5,
    'amd ryzen 7 8840': 5, 'amd ryzen 7 8840h': 5,
    'amd ryzen 7 8840hs': 5, 'amd ryzen 7 8840hx': 5,
    'amd ryzen 7 8845': 5, 'amd ryzen 7 8845hs': 5,
    'amd ryzen 9 8940': 5, 'amd ryzen 9 8940hx': 5,
    'amd ryzen 9 8945': 5, 'amd ryzen 9 8945hs': 5,
    'amd ryzen 9 9955hx': 5,
    'amd ryzen ai 5 220': 5, 'amd ryzen ai 5 330': 5,
    'amd ryzen ai 5 340': 5,
    'amd ryzen ai 7 250': 5, 'amd ryzen ai 7 260': 5,
    'amd ryzen ai 7 350': 5, 'amd ryzen ai 7 784': 5,
    'amd ryzen ai 9 270': 5, 'amd ryzen ai 9 365': 5,
    'amd ryzen ai 9 hx370': 5, 'amd ryzen ai 9 hx375': 5,
    'amd ryzen ai z2': 5,
    'apple m2 pro': 5, 'apple m3': 5,
    'intel core intel ultra 7 235u': 5,

    # ── Tier 6: En üst ──
    'intel core intel ultra 7 165h': 6, 'intel core intel ultra 7 165u': 6,
    'intel core intel ultra 9 275hx': 6, 'intel core intel ultra 9 285h': 6,
    'intel core intel ultra 9 285hx': 6,
    'apple m3 pro': 6,
}

df['cpu_priority'] = df['cpu'].map(cpu_mapping)

In [14]:
df.to_csv("/work/data/processed/01_extract_cpu_features.csv", index=False)